In [12]:
import kagglehub

# Download latest version
output_dir='./data'
# Download latest version
path = kagglehub.competition_download('titanic',output_dir=output_dir)

print("Path to competition files:", path)

100%|██████████| 34.1k/34.1k [00:00<00:00, 433kB/s]

Extracting files...
Path to competition files: ./data


In [2]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("./data/train.csv")
df_test=pd.read_csv('./data/test.csv')

print(df_train.shape)
print(df_train.head())
print(df_train["Survived"].value_counts())
print(df_train.isnull().sum())

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

In [5]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline,Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier,RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix

df=pd.read_csv('./data/train.csv')
features=['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']
x=df[features]
y=df['Survived']

numeric_features=["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_transformer=Pipeline([('imputer',SimpleImputer(strategy='median')),("scaler", StandardScaler())])
categorical_transformer=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),("encoder", OneHotEncoder(handle_unknown='ignore'))])

preprocessor=ColumnTransformer([('num',numeric_transformer,numeric_features),('cate',categorical_transformer,categorical_features)])
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", x_train.shape[0])
print("Validation rows:", x_test.shape[0])


Training rows: 712
Validation rows: 179


In [6]:
logistic=make_pipeline(preprocessor,LogisticRegression(max_iter=50))
tree=make_pipeline(preprocessor,DecisionTreeClassifier(max_depth=5,min_samples_leaf=5,random_state=42))
forest=make_pipeline(preprocessor,RandomForestClassifier(n_estimators=300,min_samples_leaf=2,random_state=42))

voting_hard=VotingClassifier(estimators=[('lr',logistic),('tree',tree),('forest',forest)],voting="hard")
voting_soft=VotingClassifier(estimators=[('lr',logistic),('tree',tree),('forest',forest)],voting="soft",weights=[1,1,2])

models = {
    "Logistic Regression": logistic,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Hard Voting": voting_hard,
    "Soft Voting": voting_soft
}

for name, model in models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)

    print(f"\n{name}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")



Logistic Regression
Accuracy: 0.8045

Decision Tree
Accuracy: 0.7542

Random Forest
Accuracy: 0.8045

Hard Voting
Accuracy: 0.8045

Soft Voting
Accuracy: 0.8045
